# V3: ReAct Agent (Tool Use — actually agentic)

V1 (router) and V2 (planner) are fixed pipelines: each stage runs in a predetermined order regardless of the input. **V3 is genuinely agentic** — the LLM picks *which* tool to call, in *what* order, based on what it has observed so far. It can iterate, change its mind, and only terminates when it decides it has enough information.

Tools available to V3:

| Tool | Returns |
| --- | --- |
| `get_recent_deploys(service, window_minutes)` | list of recent deploys with SHAs and authors |
| `query_logs(service, severity, limit)` | sample log lines at the requested severity |
| `check_pod_status(deployment)` | desired/ready replicas, restart count, last restart reason |
| `get_metric(metric, service, window_minutes)` | current vs. baseline value |
| `search_runbooks(query, k)` | BM25 hits over the runbook library |
| `propose_plan(category, primary_runbook, steps, summary)` | **terminal** — emits the final response plan |

All non-runbook tools return *mocked-but-realistic* data so the notebook stays deterministic and free to run without hitting any real production system.

In [ ]:
import os, sys, json
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')
assert os.environ.get('OPENAI_API_KEY'), 'set OPENAI_API_KEY in ../.env'

from src import ReactAgent, RunbookIndex

index = RunbookIndex(runbook_dir='../data/runbooks')
agent = ReactAgent(index=index, max_iterations=6)
print(f'agent ready. {len(index.runbooks)} runbooks indexed.')

## 1. Single run — watch the agent think

Look at the trace. The agent will choose tools, see results, then reach a plan.

In [ ]:
alert = (
    'p99 latency on the /api/checkout endpoint has spiked to 1.8s '
    '(baseline ~250ms) over the last 12 minutes. Error rate is normal. '
    'No obvious infra alarms.'
)
run = agent.run(alert)
print(ReactAgent.format_trace(run))

## 2. Several alerts — how the trajectory changes by input

The agent should follow visibly different paths for different alert categories:

- A k8s pressure alert should drive `check_pod_status` early.
- A leaked-credential alert should jump straight to `search_runbooks` and skip log queries.
- A 5xx surge should pull recent deploys and error logs first.

In [ ]:
alerts = [
    'k8s node ip-10-0-3-44 condition MemoryPressure=True. 8 pods evicted from the app-pool deployment in the last 5 minutes.',
    'GitGuardian: AWS access key AKIA*** committed to a public repo akhilg-9/sandbox four minutes ago.',
    '5xx error rate on /api/checkout is 12% over the last 10m. NullPointerException in PaymentResolver.',
    'Airflow DAG nightly_user_metrics failed: dbt test unique_users_user_id reported 47 duplicates.',
]
for a in alerts:
    run = agent.run(a)
    print('=' * 80)
    print(ReactAgent.format_trace(run))

## 3. Compare V2 vs. V3 on the same alert

Same input. V2 (pipeline) always runs route → retrieve → plan. V3 (agent) decides for itself which signals to gather.

In [ ]:
from src import RouterAgent, PlannerAgent

router = RouterAgent(prompt_version='improved')
planner = PlannerAgent(router=router, index=index, prompt_version='improved')

alert = 'Customer reports of slow page loads. p99 latency on the search API has climbed from 180ms to 1.4s starting around 09:32. No deploy in the last hour.'

print('--- V2 (pipeline) ---')
plan_v2 = planner.plan(alert)
print(f'category={plan_v2.category.value}  runbook={plan_v2.primary_runbook}')
for i, s in enumerate(plan_v2.steps, 1):
    print(f'  {i}. {s.step}')

print()
print('--- V3 (agent) ---')
run_v3 = agent.run(alert)
print(ReactAgent.format_trace(run_v3))

## What makes V3 "actually agentic"

1. **Tool use** — the LLM calls real functions, not just generates text.
2. **Decision-making over the loop** — the model decides which tool comes next from observations, not from a hard-coded pipeline.
3. **Termination decision** — the agent decides *when* it has enough information by calling `propose_plan`.

This sits at a higher rung of the autonomy ladder than V2. Higher rungs (not in this repo) would let the agent *execute* the plan against real systems — opening a Jira ticket, rolling back a deploy, draining a node — rather than only proposing one.